<a href="https://colab.research.google.com/github/nkrao8506/learn/blob/main/openenv_smapleipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
# 1. CENTRAL TENDENCY
def central_tendency(data):
    results = {}

    # Convert to numpy array if needed
    arr = np.array(data)

    # Mean (for interval, ratio, discrete, continuous)
    results['mean'] = np.mean(arr) if np.issubdtype(arr.dtype, np.number) else None

    # Median (for ordinal, interval, ratio, discrete, continuous)
    try:
        results['median'] = np.median(arr.astype(float))
    except:
        results['median'] = None

    # Mode (for all types)
    values, counts = np.unique(arr, return_counts=True)
    results['mode'] = values[np.argmax(counts)]

    # Geometric mean (for ratio data - no zeros/negatives)
    if np.all(arr > 0):
        results['geometric_mean'] = np.exp(np.mean(np.log(arr.astype(float))))

    # Trimmed mean (for continuous data - remove outliers)
    sorted_arr = np.sort(arr.astype(float))
    trim = int(0.1 * len(sorted_arr))  # 10% trim
    results['trimmed_mean'] = np.mean(sorted_arr[trim:-trim]) if len(sorted_arr) > 10 else results['mean']

    return results

In [ ]:
# 2. FIND-S AND CANDIDATE ELIMINATION ALGORITHM
class ConceptLearning:
    def __init__(self, attributes):
        self.attributes = attributes  # List of possible values for each attribute
        self.n_attributes = len(attributes)

    def find_s(self, examples):
        # Initialize most specific hypothesis
        hypothesis = ['∅'] * self.n_attributes

        for instance, label in examples:
            if label == 1:  # Positive example
                for i in range(self.n_attributes):
                    if hypothesis[i] == '∅':
                        hypothesis[i] = instance[i]
                    elif hypothesis[i] != instance[i]:
                        hypothesis[i] = '?'  # Generalize

        return hypothesis

    def candidate_elimination(self, examples):
        # Initialize most specific and most general hypotheses
        S = ['∅'] * self.n_attributes
        G = [['?' for _ in range(self.n_attributes)]]

        for instance, label in examples:
            if label == 1:  # Positive example
                # Remove inconsistent hypotheses from G
                G = [g for g in G if self._consistent(g, instance, 1)]

                # Generalize S to cover instance
                for i in range(self.n_attributes):
                    if S[i] == '∅':
                        S[i] = instance[i]
                    elif S[i] != instance[i] and S[i] != '?':
                        S[i] = '?'

                # Remove from G any hypothesis more specific than S
                G = [g for g in G if self._more_general(g, S) or g == S]

            else:  # Negative example
                # Remove inconsistent hypotheses from S
                if self._consistent(S, instance, 0):
                    # Specialize G to exclude instance
                    new_G = []
                    for g in G:
                        if not self._consistent(g, instance, 0):
                            new_G.append(g)
                        else:
                            # Generate specializations
                            for i in range(self.n_attributes):
                                if g[i] == '?':
                                    for val in self.attributes[i]:
                                        if val != instance[i]:
                                            new_g = g.copy()
                                            new_g[i] = val
                                            if self._consistent(new_g, instance, 0):
                                                new_G.append(new_g)
                    G = self._remove_subsumed(new_G)

        return S, G

    def _consistent(self, hypothesis, instance, label):
        matches = all(h == '?' or h == x for h, x in zip(hypothesis, instance))
        return matches if label == 1 else not matches

    def _more_general(self, h1, h2):
        return all(a == '?' or a == b for a, b in zip(h1, h2))

    def _remove_subsumed(self, hypotheses):
        unique = []
        for h in hypotheses:
            if not any(self._more_general(other, h) and other != h for other in hypotheses):
                if h not in unique:
                    unique.append(h)
        return unique

In [ ]:
# 3. BACKPROPAGATION (Neural Network)
class NeuralNetwork:
    def __init__(self, layers, learning_rate=0.01):
        self.lr = learning_rate
        self.layers = layers
        self.weights = []
        self.biases = []

        # Initialize weights and biases
        for i in range(len(layers) - 1):
            self.weights.append(np.random.randn(layers[i], layers[i+1]) * 0.01)
            self.biases.append(np.zeros((1, layers[i+1])))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def sigmoid_derivative(self, x):
        return x * (1 - x)

    def forward(self, X):
        self.activations = [X]
        self.z_values = []

        current = X
        for w, b in zip(self.weights, self.biases):
            z = np.dot(current, w) + b
            self.z_values.append(z)
            current = self.sigmoid(z)
            self.activations.append(current)

        return current

    def backward(self, X, y):
        m = X.shape[0]
        deltas = []

        # Output layer error
        error = self.activations[-1] - y
        delta = error * self.sigmoid_derivative(self.activations[-1])
        deltas.append(delta)

        # Hidden layers
        for i in range(len(self.weights) - 1, 0, -1):
            error = np.dot(deltas[-1], self.weights[i].T)
            delta = error * self.sigmoid_derivative(self.activations[i])
            deltas.append(delta)

        deltas.reverse()

        # Update weights and biases
        for i in range(len(self.weights)):
            self.weights[i] -= self.lr * np.dot(self.activations[i].T, deltas[i]) / m
            self.biases[i] -= self.lr * np.sum(deltas[i], axis=0, keepdims=True) / m

    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            output = self.forward(X)
            self.backward(X, y)

            if epoch % 100 == 0:
                loss = np.mean((output - y) ** 2)
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def predict(self, X):
        return self.forward(X)

In [ ]:
# 4. K-MEANS CLUSTERING
class KMeans:
    def __init__(self, k=3, max_iters=100, tol=1e-4):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol

    def fit(self, X):
        n_samples, n_features = X.shape

        # Initialize centroids randomly
        np.random.seed(42)
        random_idx = np.random.choice(n_samples, self.k, replace=False)
        self.centroids = X[random_idx].copy()

        for iteration in range(self.max_iters):
            # Assign points to nearest centroid
            distances = self._compute_distances(X)
            labels = np.argmin(distances, axis=1)

            # Update centroids
            new_centroids = np.array([X[labels == i].mean(axis=0)
                                      for i in range(self.k)])

            # Check convergence
            if np.all(np.abs(new_centroids - self.centroids) < self.tol):
                break

            self.centroids = new_centroids

        self.labels_ = labels
        return self

    def _compute_distances(self, X):
        """Compute Euclidean distance from each point to each centroid"""
        distances = np.zeros((X.shape[0], self.k))
        for i in range(self.k):
            distances[:, i] = np.sqrt(np.sum((X - self.centroids[i]) ** 2, axis=1))
        return distances

    def predict(self, X):
        distances = self._compute_distances(X)
        return np.argmin(distances, axis=1)

In [ ]:
# 5. KNN ALGORITHM
class KNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def predict(self, X):
        predictions = []
        for x in X:
            # Compute distances to all training points
            distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))

            # Get k nearest neighbors
            k_indices = np.argsort(distances)[:self.k]
            k_nearest_labels = self.y_train[k_indices]

            # Majority vote
            unique, counts = np.unique(k_nearest_labels, return_counts=True)
            predictions.append(unique[np.argmax(counts)])

        return np.array(predictions)

    def predict_proba(self, X):
        probabilities = []
        classes = np.unique(self.y_train)

        for x in X:
            distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
            k_indices = np.argsort(distances)[:self.k]
            k_nearest_labels = self.y_train[k_indices]

            # Calculate probabilities
            probs = []
            for c in classes:
                probs.append(np.sum(k_nearest_labels == c) / self.k)
            probabilities.append(probs)

        return np.array(probabilities)

In [ ]:
# 6. DECISION TREES
class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2, criterion='entropy'):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion  # 'entropy' or 'gini'

    def entropy(self, y):
        proportions = np.bincount(y) / len(y)
        return -np.sum([p * np.log2(p) for p in proportions if p > 0])

    def gini(self, y):
        proportions = np.bincount(y) / len(y)
        return 1 - np.sum(proportions ** 2)

    def information_gain(self, X_column, y, threshold):
        parent_impurity = self.entropy(y) if self.criterion == 'entropy' else self.gini(y)

        # Create split
        left_mask = X_column <= threshold
        right_mask = X_column > threshold

        if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
            return 0

        # Calculate weighted impurity of children
        n = len(y)
        n_left, n_right = np.sum(left_mask), np.sum(right_mask)

        if self.criterion == 'entropy':
            left_impurity = self.entropy(y[left_mask])
            right_impurity = self.entropy(y[right_mask])
        else:
            left_impurity = self.gini(y[left_mask])
            right_impurity = self.gini(y[right_mask])

        child_impurity = (n_left / n) * left_impurity + (n_right / n) * right_impurity

        return parent_impurity - child_impurity

    def best_split(self, X, y):
        best_gain = -1
        best_feature = None
        best_threshold = None

        n_features = X.shape[1]

        for feature in range(n_features):
            X_column = X[:, feature]
            thresholds = np.unique(X_column)

            for threshold in thresholds:
                gain = self.information_gain(X_column, y, threshold)

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))

        # Stopping criteria
        if (depth >= self.max_depth or
            n_classes == 1 or
            n_samples < self.min_samples_split):
            return {'type': 'leaf', 'class': np.bincount(y).argmax()}

        # Find best split
        feature, threshold = self.best_split(X, y)

        if feature is None:
            return {'type': 'leaf', 'class': np.bincount(y).argmax()}

        # Split data
        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold

        # Recursively build left and right subtrees
        left_subtree = self.build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self.build_tree(X[right_mask], y[right_mask], depth + 1)

        return {
            'type': 'internal',
            'feature': feature,
            'threshold': threshold,
            'left': left_subtree,
            'right': right_subtree
        }

    def fit(self, X, y):
        self.tree = self.build_tree(X, y)
        self.classes_ = np.unique(y)
        return self

    def predict_sample(self, x, node):
        if node['type'] == 'leaf':
            return node['class']

        if x[node['feature']] <= node['threshold']:
            return self.predict_sample(x, node['left'])
        else:
            return self.predict_sample(x, node['right'])

    def predict(self, X):
        return np.array([self.predict_sample(x, self.tree) for x in X])

    def print_tree(self, node=None, indent=""):
        if node is None:
            node = self.tree

        if node['type'] == 'leaf':
            print(f"{indent}Leaf: Class {node['class']}")
        else:
            print(f"{indent}Feature[{node['feature']}] <= {node['threshold']:.2f}")
            print(f"{indent}Left:")
            self.print_tree(node['left'], indent + "  ")
            print(f"{indent}Right:")
            self.print_tree(node['right'], indent + "  ")

In [ ]:
# QUICK TEST FUNCTION
def run_tests():
    print("1. CENTRAL TENDENCY")
    data = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
    print(f"Data: {data}")
    print(central_tendency(data))

    print("\n2. FIND-S AND CANDIDATE ELIMINATION")
    attributes = [['sunny', 'cloudy', 'rainy'], ['warm', 'cold'], ['high', 'normal']]
    clf = ConceptLearning(attributes)
    examples = [
        (['sunny', 'warm', 'high'], 1),
        (['sunny', 'warm', 'normal'], 1),
        (['rainy', 'cold', 'high'], 0)
    ]
    print("Examples:", examples)
    print("Find-S Hypothesis:", clf.find_s(examples))
    S, G = clf.candidate_elimination(examples)
    print(f"S: {S}, G: {G}")

    print("\n3. BACKPROPAGATION (XOR Problem)")
    X = np.array([[0.8, 0], [0.991, 1], [1.001, 0], [1.29, 1]])
    y = np.array([[0], [1], [1], [0]])
    nn = NeuralNetwork([2, 4, 1], learning_rate=0.5)
    nn.train(X, y, epochs=2000)
    predictions = nn.predict(X)
    print(f"Predictions:\n{predictions.round(3)}")

    print("\n4. K-MEANS CLUSTERING")
    np.random.seed(42)
    X = np.vstack([
        np.random.randn(20, 2) + [0, 0],
        np.random.randn(20, 2) + [5, 5],
        np.random.randn(20, 2) + [0, 5]
    ])
    kmeans = KMeans(k=3)
    kmeans.fit(X)
    print(f"Centroids:\n{kmeans.centroids}")
    print(f"Labels distribution: {np.bincount(kmeans.labels_)}")

    print("\n5. KNN ALGORITHM")
    X_train = np.array([[1, 2], [2, 3], [3, 4], [6, 7], [7, 8], [8, 9]])
    y_train = np.array([0, 0, 0, 1, 1, 1])
    knn = KNN(k=3)
    knn.fit(X_train, y_train)
    X_test = np.array([[2, 2], [7, 7]])
    predictions = knn.predict(X_test)
    print(f"Test points: {X_test.tolist()}")
    print(f"Predictions: {predictions}")

    print("\n6. DECISION TREE")
    X = np.array([[2.5], [3.5], [4.5], [5.5], [6.5], [7.5]])
    y = np.array([0, 0, 0, 1, 1, 1])
    dt = DecisionTree(max_depth=3)
    dt.fit(X, y)
    dt.print_tree()
    print(f"Predictions: {dt.predict(X)}")

run_tests()

1. CENTRAL TENDENCY
Data: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
{'mean': np.float64(55.0), 'median': np.float64(55.0), 'mode': np.int64(10), 'geometric_mean': np.float64(45.28728688116765), 'trimmed_mean': np.float64(55.0)}

2. FIND-S AND CANDIDATE ELIMINATION
Examples: [(['sunny', 'warm', 'high'], 1), (['sunny', 'warm', 'normal'], 1), (['rainy', 'cold', 'high'], 0)]
Find-S Hypothesis: ['sunny', 'warm', '?']
S: ['sunny', 'warm', '?'], G: [['?', '?', '?']]

3. BACKPROPAGATION (XOR Problem)
Epoch 0, Loss: 0.2500
Epoch 100, Loss: 0.2500
Epoch 200, Loss: 0.2500
Epoch 300, Loss: 0.2500
Epoch 400, Loss: 0.2500
Epoch 500, Loss: 0.2500
Epoch 600, Loss: 0.2500
Epoch 700, Loss: 0.2500
Epoch 800, Loss: 0.2500
Epoch 900, Loss: 0.2500
Epoch 1000, Loss: 0.2500
Epoch 1100, Loss: 0.2500
Epoch 1200, Loss: 0.2500
Epoch 1300, Loss: 0.2500
Epoch 1400, Loss: 0.2500
Epoch 1500, Loss: 0.2500
Epoch 1600, Loss: 0.2500
Epoch 1700, Loss: 0.2500
Epoch 1800, Loss: 0.2500
Epoch 1900, Loss: 0.2500
Predictions:
[